# Lab | Spotify API (SpotiPy)

**Important note:** the `related_artists` endpoint was **deprecated by Spotify on November 27, 2024** (returns a 403 error for any app created after that date). The same applies to `featured_playlists`. This notebook therefore provides:
- the code "expected" by the exercise (function, structure, requested logic),
- a **working fallback** based on `sp.search()` by genre / keyword, which replaces these deprecated endpoints.



## Authentication and API initialization

In [1]:
!pip install spotipy --quiet

In [2]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

CLIENT_ID = "<your_client_id>"
CLIENT_SECRET = "<your_client_secret>"

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
)

## Exercise 1: Discovering New Music through Your Favorite Artists

### 1. List Your Favorite Artists

In [3]:
artists = ["Los Fabulosos Cadillacs", "Manu Chao", "Muchachito Bombo Infierno"]

### 2. Fetch Top Tracks

In [4]:
def get_top_tracks(artist_name, limit=5, country="US"):
    """
    Return the names of the top `limit` tracks for a given artist name.
    """
    # Find the artist to get their Spotify ID
    result = sp.search(q=f"artist:{artist_name}", type="artist", limit=1)
    items = result["artists"]["items"]
    if not items:
        return []
    artist_id = items[0]["id"]

    top_tracks = sp.artist_top_tracks(artist_id, country=country)
    return [track["name"] for track in top_tracks["tracks"][:limit]]


top_tracks_list = [get_top_tracks(artist) for artist in artists]
top_tracks_list

SpotifyOauthError: error: invalid_client, error_description: Invalid client

### 3. Discover Related Artists

Since `sp.artist_related_artists()` is deprecated (403 for new apps), we use a **genre-based fallback**: fetch the artist's main genre, then search for other artists sharing that genre.

In [ ]:
def find_related_artists(artist_name, limit=5):
    """
    Return up to `limit` artist names related to the given artist.

    Primary approach (works only on pre-Nov-2024 Extended Quota apps):
        sp.artist_related_artists(artist_id)

    Fallback (works for all apps): search by the artist's main genre.
    """
    result = sp.search(q=f"artist:{artist_name}", type="artist", limit=1)
    items = result["artists"]["items"]
    if not items:
        return []
    artist = items[0]
    artist_id = artist["id"]

    # --- Original approach (kept for reference, will raise 403 on new apps) ---
    # related = sp.artist_related_artists(artist_id)
    # return [a["name"] for a in related["artists"][:limit]]

    # --- Fallback: search by genre ---
    genres = artist.get("genres", [])
    if not genres:
        return []
    genre = genres[0]
    search_result = sp.search(q=f'genre:"{genre}"', type="artist", limit=limit + 1)
    related_names = [
        a["name"] for a in search_result["artists"]["items"]
        if a["name"] != artist_name
    ]
    return related_names[:limit]


related_artists_list = [find_related_artists(artist) for artist in artists]
related_artists_list

### Challenge: combine top tracks of favorite artists + top tracks of related artists

In [ ]:
playlist = []

for artist in artists:
    playlist.extend(get_top_tracks(artist))
    for related_artist in find_related_artists(artist):
        playlist.extend(get_top_tracks(related_artist))

# Remove duplicates while keeping order
playlist = list(dict.fromkeys(playlist))
playlist

## Exercise 2: Unraveling the World of Playlists

### 1. Featured Exploration

`sp.featured_playlists()` is also deprecated for new apps. We therefore use a **search over public playlists** as a fallback (e.g. with a generic keyword like `"Top Hits"`).

In [ ]:
def get_top_playlists(query="Top Hits", limit=5):
    """
    Return name and id of `limit` playlists.

    Primary approach (deprecated on new apps):
        sp.featured_playlists()

    Fallback: search public playlists by keyword.
    """
    # --- Original approach (kept for reference) ---
    # featured = sp.featured_playlists()
    # playlists = featured["playlists"]["items"][:limit]

    # --- Fallback ---
    result = sp.search(q=query, type="playlist", limit=limit)
    playlists = result["playlists"]["items"]

    return [(p["name"], p["id"]) for p in playlists if p]


top_playlists = get_top_playlists()
top_playlists

### 2. Deep Dive: choose one playlist and display its details

In [ ]:
chosen_playlist_id = top_playlists[0][1]

playlist = sp.playlist(chosen_playlist_id)

print(playlist["name"])
print(playlist["description"])
print(playlist["tracks"]["total"])

### 3. Track-tastic: names of the first 10 tracks

In [ ]:
tracks = sp.playlist_tracks(chosen_playlist_id, limit=10)

track_names = [item["track"]["name"] for item in tracks["items"] if item["track"]]
track_names

### 4. Artistic Flair: dictionary of track name -> list of artist names

In [ ]:
track_artists_dict = {}

for item in tracks["items"]:
    track = item["track"]
    if not track:
        continue
    track_artists_dict[track["name"]] = [artist["name"] for artist in track["artists"]]

track_artists_dict